# Building an AI Agent from ScratchAn **agent** is a language model wired into a loop that can act on the world.The model itself cannot do anything. It can only look at the tools you describeto it and reply *"call this function with these arguments"*. Your code reads thatreply, runs the function, and hands the result back. That loop is the agent.By the end of this notebook you will have written that loop yourself, and watchedit find and fix a bug in a real file on your disk.**Setup:** you need an [OpenRouter](https://openrouter.ai/) API key. Keep it handy.

In [ ]:
%pip install --quiet openai pytestimport jsonimport subprocessimport sysfrom getpass import getpassfrom pathlib import Pathfrom openai import OpenAIMODEL = "nvidia/nemotron-3.5-lightning"WORKDIR = Path.cwd()# getpass keeps the key out of the notebook file and out of your terminal historyclient = OpenAI(    base_url="https://openrouter.ai/api/v1",    api_key=getpass("OpenRouter API key: "),)print("model   :", MODEL)print("workdir :", WORKDIR)

## 1. Does the API work?Before anything else, prove the connection. One request, one response, no tools.This is an ordinary HTTPS call: your messages go out, some text comes back. Nothingis running on your machine and nothing is running on theirs except the model.

In [ ]:
response = client.chat.completions.create(    model=MODEL,    messages=[{"role": "user", "content": "Write a haiku about debugging code."}],    max_tokens=100,)haiku = response.choices[0].message.contentprint(haiku)print("\ntokens in :", response.usage.prompt_tokens)print("tokens out:", response.usage.completion_tokens)

## 2. The taskWe are going to build an agent that fixes a failing test.The next two cells write the files it will work on. Read `buggy.py` carefully anddecide for yourself where the bug is before you run anything.> **Re-run the `%%writefile buggy.py` cell at any time to put the bug back** and try> the agent again from a clean start.

In [ ]:
%%writefile buggy.pydef add_reading(reading, log=[]):    """Append a sensor reading to a log and return the log."""    log.append(reading)    return logdef average(readings):    """Return the mean of a list of readings."""    return sum(readings) / len(readings)

In [ ]:
%%writefile test_buggy.pyfrom buggy import add_reading, averagedef test_average():    assert average([2, 4, 6]) == 4def test_logs_are_independent():    first = add_reading(1)    second = add_reading(2)    assert first == [1]    assert second == [2]

In [ ]:
# Run the tests yourself first, so you know what the agent is up against.print(subprocess.run(    [sys.executable, "-m", "pytest", "-q"],    capture_output=True, text=True, cwd=WORKDIR,).stdout)

## 3. The toolsA "tool" is nothing exotic: **a Python function you wrote, plus a JSON descriptionof it that you send along with the prompt.**Three functions, about ten lines. Note that they return error *strings* rather thanraising — the agent reads those errors and tries again, which is a large part of whythe loop works at all.`run_tests` deliberately shells out to a fresh Python process. That avoids Python'smodule cache, which would otherwise keep serving the old, unfixed `buggy.py`.

In [ ]:
def read_file(path: str) -> str:    return (WORKDIR / path).read_text()def edit_file(path: str, old: str, new: str) -> str:    p = WORKDIR / path    text = p.read_text()    if text.count(old) == 0:        return "ERROR: 'old' not found in the file. Read it again and match it exactly."    if text.count(old) > 1:        return "ERROR: 'old' appears more than once. Include more surrounding context."    p.write_text(text.replace(old, new))    return f"ok, edited {path}"def run_tests() -> str:    result = subprocess.run(        [sys.executable, "-m", "pytest", "-q"],        capture_output=True, text=True, timeout=60, cwd=WORKDIR,    )    return (result.stdout + result.stderr)[-2000:] or "(no output)"# The dispatch table. This dict is the agent's entire set of powers:# if it is not in here, the model cannot do it.DISPATCH = {    "read_file": read_file,    "edit_file": edit_file,    "run_tests": run_tests,}

## 4. Describing the tools to the modelThe model never sees the function bodies — only these descriptions. So the wordinghere is prompt engineering, not documentation: a vague description produces a modelthat misuses the tool.

In [ ]:
TOOLS = [    {        "type": "function",        "function": {            "name": "read_file",            "description": "Read the full contents of a file in the working directory.",            "parameters": {                "type": "object",                "properties": {                    "path": {"type": "string", "description": "e.g. buggy.py"},                },                "required": ["path"],            },        },    },    {        "type": "function",        "function": {            "name": "edit_file",            "description": (                "Replace an exact snippet of text in a file. The 'old' string must "                "appear exactly once in the file, whitespace included."            ),            "parameters": {                "type": "object",                "properties": {                    "path": {"type": "string"},                    "old": {"type": "string", "description": "Text to replace."},                    "new": {"type": "string", "description": "Replacement text."},                },                "required": ["path", "old", "new"],            },        },    },    {        "type": "function",        "function": {            "name": "run_tests",            "description": "Run the pytest suite and return its output.",            "parameters": {"type": "object", "properties": {}},        },    },]

In [ ]:
SYSTEM = (    "You are a debugging assistant. Use the tools to inspect and fix the code. "    "Always run the tests after making an edit, and keep going until they pass. "    "When they pass, reply with one sentence explaining what the bug was.")TASK = "The tests in test_buggy.py are failing. Find the bug in buggy.py and fix it."def assistant_message(msg):    """Turn the SDK's response object back into a plain message dict."""    return {        "role": "assistant",        "content": msg.content or "",        "tool_calls": [            {                "id": c.id,                "type": "function",                "function": {                    "name": c.function.name,                    "arguments": c.function.arguments,                },            }            for c in msg.tool_calls        ],    }

## 5. The loop — this is the part you writeEverything so far has been scaffolding. The agent is this:```send messages + tool descriptions  ->  model                                   <-  "call read_file with {path: buggy.py}"run the function yourselfappend the result to messagesrepeat```Four things to get right:1. Call the model with `messages` **and** `tools`.2. If the reply has no `tool_calls`, the agent is done — print it and stop.3. Otherwise append the assistant message, then run each requested tool.4. Append each result as a message with `role="tool"` and the matching `tool_call_id`.The `max_turns` cap is not optional. Without it a confused agent loops until yourcredit runs out.

In [ ]:
def run_agent(task, max_turns=8, tools=None):    tools = tools or TOOLS    messages = [        {"role": "system", "content": SYSTEM},        {"role": "user", "content": task},    ]    for turn in range(1, max_turns + 1):        print(f"\n--- turn {turn} ---")        reply = client.chat.completions.create(            model=MODEL,            messages=messages,            tools=tools,            max_tokens=1024,        )        msg = reply.choices[0].message        if not msg.tool_calls:            print("FINAL:", msg.content)            return msg.content        messages.append(assistant_message(msg))        for call in msg.tool_calls:            args = json.loads(call.function.arguments)            print(f"  -> {call.function.name}({args})")            output = DISPATCH[call.function.name](**args)      # <-- YOUR code runs it            print(f"  <- {output[:300]}")            messages.append({                "role": "tool",                "tool_call_id": call.id,                "content": output,            })    return "hit the turn cap without finishing"

## 6. Run itWatch the transcript. Every `->` is the model asking for something and every `<-`is your own Python handing back a result.

In [ ]:
result = run_agent(TASK)

## 7. Did it actually change the file?It did. This is the moment worth pausing on: a language model, which can only emittext, has modified a file on your disk — because you wrote the four lines that let it.

In [ ]:
print(Path("buggy.py").read_text())print(run_tests())

## 8. Things to notice- **The dispatch table is a permission boundary.** The agent can do exactly what is  in that dict and nothing else. Add `run_shell` to it and you have handed a  probabilistic system your terminal.- **The API is stateless.** The provider remembers nothing between calls; `messages`  grows on your side and the whole history is resent every turn. That is why a  six-turn agent costs far more than six single calls.- **The model never saw the function bodies.** Only the descriptions in `TOOLS`.- **It is not deterministic.** Reset the bug (re-run the `%%writefile buggy.py` cell)  and run it again. The path it takes will differ, and sometimes the number of turns will too.### Try this1. Delete `run_tests` from `DISPATCH` and `TOOLS`, then run the agent again.   It can still read and edit, but it can no longer check its work. What changes?2. Tell the agent the bug is in `average()` instead. Does it agree with you, or   with the evidence?

---# Instructor notes**Timing (~80 minutes)**| Time | Segment ||---|---|| 0:00 | Setup check, cells 1–2. Have them run the haiku cell before you talk. || 0:10 | Explain tools and the loop at the whiteboard. Cells 3–4 are read-only. || 0:25 | They write cell 5. This is the long one. Expect 20–30 minutes. || 0:50 | Run it. Walk the transcript together. || 1:00 | The two flips below, plus discussion. |**The bug.** `log=[]` is evaluated once at function definition, so both calls shareone list. It reads as completely correct and fails only on the second call — which isthe entire point of the exercise. It cannot be found by staring; the agent has to runthe tests and reason from evidence. `test_average` passes throughout, so the agentalso has to work out *which* test is failing.**Flip 1 — remove the evidence.**```pythonTOOLS_NO_TESTS = [t for t in TOOLS if t["function"]["name"] != "run_tests"]del DISPATCH["run_tests"]run_agent(TASK, tools=TOOLS_NO_TESTS)```The agent now patches confidently and blindly. Some runs will "fix" the wrongfunction. This is the argument for verification, demonstrated rather than asserted.Restore with `DISPATCH["run_tests"] = run_tests`.**Flip 2 — the reward hack.** Nothing currently stops the agent editing`test_buggy.py`. Some runs will make the failure disappear by weakening the assertionor special-casing the input rather than fixing `add_reading`. If it happens, stop andask the room what went wrong. The framing for a data-science cohort is one line:*that is fitting on the test set.* Then add the guard and show that it is four linesof their own code, not a property of the model:```pythondef edit_file_guarded(path, old, new):    if path.startswith("test_"):        return "ERROR: the test files are read-only."    return edit_file(path, old, new)DISPATCH["edit_file"] = edit_file_guarded```**Flip 3 — green is not correct.** `average([])` raises `ZeroDivisionError` and notest covers it. The agent will report success with that bug still present. Ask whatelse might be broken that the suite never asked about.**On the model.** `nvidia/nemotron-3.5-lightning` is a 30B mixture-of-experts modelwith ~3B active parameters, trained specifically for tool-calling execution ratherthan for deep reasoning, at roughly $0.08 / $0.20 per million input / output tokens.That makes it very cheap for a room of thirty and a genuinely good fit for themechanics of this lab. It also means diagnosis is its weaker skill, which is afeature here — it leans harder on `run_tests`, and it makes the reward hack in Flip 2more likely to appear. Run the notebook end to end the week before the session. If itstruggles to converge within eight turns, raise the cap to 12 or change one string:```pythonMODEL = "openai/gpt-4o-mini"        # or any other OpenRouter slug```**Cost.** A full run is a handful of turns of a few thousand tokens. Thirty studentsrunning it several times each will not trouble a $5 credit. Still, set a spend limiton the key and use a workshop-scoped key you rotate afterwards.